In [ ]:
#### Generates feature importance + permutation importance

# rf_models/
# ├── config.py          # run configs as dicts or dataclasses
# ├── spatial_cv.py      # spatial block splitting logic
# ├── train.py           # fit model given config + data, return results dict
# ├── evaluate.py        # metrics, plots
# ├── importance.py      # feature importance + permutation importance
# └── run.py             # orchestrates everything, saves outputs

In [ ]:
import numpy as np
import pandas as pd
from sklearn.inspection import permutation_importance
from config import RunConfig


def get_impurity_importance(model, feature_cols: list) -> pd.DataFrame:
    """
    Built-in RF feature importance (mean decrease in impurity).
    Fast but biased toward high-cardinality features.
    """
    return pd.DataFrame({
        "feature":            feature_cols,
        "impurity_importance": model.feature_importances_,
    }).sort_values("impurity_importance", ascending=False)


def get_permutation_importance(model, X_test, y_test, config: RunConfig, n_repeats: int = 10) -> pd.DataFrame:
    """
    Permutation importance on the test set.
    Slower but unbiased and reflects out-of-sample performance.
    """
    result = permutation_importance(
        model, X_test, y_test,
        n_repeats    = n_repeats,
        random_state = config.random_seed,
        n_jobs       = -1,
        scoring      = "neg_root_mean_squared_error",
    )
    return pd.DataFrame({
        "feature":                X_test.columns.tolist(),
        "permutation_importance": result.importances_mean,
        "permutation_std":        result.importances_std,
    }).sort_values("permutation_importance", ascending=False)


def get_feature_importance(results: dict, df: pd.DataFrame, config: RunConfig) -> pd.DataFrame:
    """
    Compute both importance types for each fold, then aggregate across folds.
    Returns a single DataFrame with mean and std of each importance type per feature.
    """
    X = df[results["feature_cols"]]
    y = df[config.target]

    fold_impurity     = []
    fold_permutation  = []

    for fold_result in results["fold_results"]:
        model    = fold_result["best_model"]
        test_idx = df.index.isin(
            results["predictions"]
            .loc[results["predictions"]["fold"] == fold_result["fold"]]
            .index
        )
        X_test = X.loc[test_idx]
        y_test = y.loc[test_idx]

        imp  = get_impurity_importance(model, results["feature_cols"])
        perm = get_permutation_importance(model, X_test, y_test, config)

        imp["fold"]  = fold_result["fold"]
        perm["fold"] = fold_result["fold"]

        fold_impurity.append(imp)
        fold_permutation.append(perm)

    # aggregate across folds
    impurity_df    = pd.concat(fold_impurity)
    permutation_df = pd.concat(fold_permutation)

    impurity_agg = (
        impurity_df
        .groupby("feature")["impurity_importance"]
        .agg(["mean", "std"])
        .rename(columns={"mean": "impurity_mean", "std": "impurity_std"})
        .reset_index()
    )

    permutation_agg = (
        permutation_df
        .groupby("feature")["permutation_importance"]
        .agg(["mean", "std"])
        .rename(columns={"mean": "permutation_mean", "std": "permutation_std"})
        .reset_index()
    )

    combined = impurity_agg.merge(permutation_agg, on="feature")
    combined = combined.sort_values("permutation_mean", ascending=False).reset_index(drop=True)

    print("\n── Feature Importance (top 15) ──────────────────────────────")
    print(combined.head(15).to_string(index=False))

    return combined